# Semana 7 · Validación — Clima (Estación Meteorológica del Fundo)
## Control de calidad y validación de datos

**Producto de la sesión:** dataset validado + resumen de reglas + incidencias + métricas + plan de acción.

> **Principio de trabajo:** validar no es volver a limpiar. Primero medimos, clasificamos y documentamos. No corregimos silenciosamente.

## 1. Archivo de entrada
Trabajaremos con el producto de la Semana 6: `clima_limpio.xlsx` (columnas en snake_case + flags de trazabilidad `flag_temperatura_imposible`, `flag_humedad_imposible`).

In [ ]:
# Si estás en Google Colab, esto abre el selector de archivos.
# Sube clima_limpio.xlsx cuando te lo pida.
try:
    from google.colab import files
    uploaded = files.upload()
    ARCHIVO = next(iter(uploaded))
except ImportError:
    ARCHIVO = "clima_limpio.xlsx"

print("Archivo de trabajo:", ARCHIVO)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

df = pd.read_excel(ARCHIVO)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)
display(df.head(5))
print("\nColumnas disponibles:")
print(df.columns.tolist())

## 2. Contrato estructural del dataset
Antes de validar valores, comprobamos que existan las columnas indispensables. Si falta una, se detiene el proceso — no tiene sentido calcular calidad sobre un archivo incompleto.

In [ ]:
COLUMNAS_NUCLEO = [
    "fecha", "temperatura_oc", "temperatura_minima_oc", "temperatura_maxima_oc",
    "humedad_relativa", "precipitacion_mm", "et0_mm",
    "velocidad_de_viento_promedio_km_h", "radiacion_solar_w_m2",
    "grados_dia_oc", "grados_dia_acumulado_ano_oc",
    "flag_temperatura_imposible", "flag_humedad_imposible",
]

faltan_nucleo = [c for c in COLUMNAS_NUCLEO if c not in df.columns]
if faltan_nucleo:
    raise ValueError(f"El archivo no cumple el esquema esperado. Faltan columnas: {faltan_nucleo}")

print("✓ Esquema mínimo verificado.")

## 3. Preparación exclusiva para validar
Convertir `fecha` a datetime y confirmar que las columnas numéricas y las banderas de la Semana 6 llegaron con el tipo correcto. Si faltara una bandera, no se inventa: se detiene el proceso.

In [ ]:
fecha_original = df["fecha"].copy()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
fecha_no_convertida = fecha_original.notna() & df["fecha"].isna()
if fecha_no_convertida.any():
    raise ValueError(f"Hay {int(fecha_no_convertida.sum())} fechas no interpretables. Revise la Semana 6.")

for col in ["temperatura_oc", "humedad_relativa", "precipitacion_mm", "et0_mm", "grados_dia_oc"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"La columna {col!r} debe ser numérica. Revise la Semana 6.")

FLAGS_SEMANA6 = ["flag_temperatura_imposible", "flag_humedad_imposible"]
faltan_flags = [c for c in FLAGS_SEMANA6 if c not in df.columns]
if faltan_flags:
    raise ValueError(f"Faltan banderas de trazabilidad de Semana 6: {faltan_flags}.")

for col in FLAGS_SEMANA6:
    df[col] = df[col].astype(bool)

print("✓ Fechas, variables numéricas y banderas verificadas.")

## 4. Política de severidad del caso

| Severidad | Umbral | Interpretación |
|---|---:|---|
| **CRÍTICA** | 100% | Una falla puede bloquear el uso del dataset. |
| **ALTA** | 98% | Requiere corrección prioritaria. |
| **MEDIA** | 95% | Puede permitir uso condicionado, pero debe documentarse. |
| **AVISO** | Sin umbral | Señal de revisión; no implica borrar automáticamente. |

## 5. Motor reusable de validación

In [ ]:
# Umbrales de aceptación por severidad (mismo criterio que Semana 7).
UMBRALES = {"CRÍTICA": 100.0, "ALTA": 98.0, "MEDIA": 95.0}

resumen_reglas = []
metadatos_reglas = {}

def _serie_booleana(valor, indice):
    if isinstance(valor, pd.Series):
        return valor.reindex(indice)
    return pd.Series(bool(valor), index=indice)

def registrar_regla(rule_id, dimension, campo, descripcion, severidad,
                     condicion_cumple, aplicable=None):
    indice = df.index
    cumple = _serie_booleana(condicion_cumple, indice).fillna(False).astype(bool)
    aplica = pd.Series(True, index=indice) if aplicable is None else _serie_booleana(aplicable, indice).fillna(False).astype(bool)

    incumple = aplica & (~cumple)
    col_error = f"err_{rule_id.lower()}"
    df[col_error] = incumple

    n_aplica = int(aplica.sum())
    n_incumple = int(incumple.sum())
    n_cumple = n_aplica - n_incumple
    porcentaje = round(100 * n_cumple / n_aplica, 2) if n_aplica > 0 else np.nan
    umbral = UMBRALES.get(severidad, np.nan)
    estado = "NO APLICA" if n_aplica == 0 else ("CUMPLE" if porcentaje >= umbral else "NO CUMPLE")

    resumen_reglas.append({
        "regla": rule_id, "dimension": dimension, "campo": campo,
        "descripcion": descripcion, "severidad": severidad, "umbral_pct": umbral,
        "aplicables": n_aplica, "cumplen": n_cumple, "incumplen": n_incumple,
        "cumplimiento_pct": porcentaje, "estado_regla": estado,
    })
    metadatos_reglas[rule_id] = {
        "dimension": dimension, "campo": campo, "descripcion": descripcion,
        "severidad": severidad, "col_error": col_error,
    }


## 6. Completitud

In [ ]:
registrar_regla("R01", "Completitud", "fecha", "Fecha obligatoria", "CRÍTICA", df["fecha"].notna())
registrar_regla("R02", "Completitud", "temperatura_oc", "Temperatura media obligatoria", "CRÍTICA", df["temperatura_oc"].notna())
registrar_regla("R03", "Completitud", "humedad_relativa", "Humedad relativa obligatoria", "ALTA", df["humedad_relativa"].notna())
registrar_regla("R04", "Completitud", "precipitacion_mm", "Precipitación requerida para el balance hídrico", "MEDIA", df["precipitacion_mm"].notna())
registrar_regla("R05", "Completitud", "et0_mm", "ET0 requerida para análisis agroclimático", "MEDIA", df["et0_mm"].notna())
registrar_regla("R06", "Completitud", "radiacion_solar_w_m2", "Radiación solar requerida para análisis agroclimático", "MEDIA", df["radiacion_solar_w_m2"].notna())
print("✓ Reglas R01–R06 ejecutadas.")

## 7. Unicidad

In [ ]:
fecha_no_repetida = ~df["fecha"].duplicated(keep=False)
registrar_regla("R07", "Unicidad", "fecha", "La fecha no debe repetirse (un registro por día)", "CRÍTICA",
                 fecha_no_repetida, aplicable=df["fecha"].notna())
print("✓ Regla R07 ejecutada.")

## 8. Validez: rango físico y periodo autorizado
El alcance oficial del proyecto declara el periodo 01/03/2026–31/08/2026. Una fecha fuera de esa ventana no es un "dato sucio", pero sí queda fuera del alcance analítico declarado.

In [ ]:
INICIO_ALCANCE = pd.Timestamp("2026-03-01")
FIN_ALCANCE = pd.Timestamp("2026-08-31")

registrar_regla("R08", "Validez", "fecha", "Fecha dentro del periodo autorizado del proyecto", "ALTA",
                 df["fecha"].between(INICIO_ALCANCE, FIN_ALCANCE), aplicable=df["fecha"].notna())

registrar_regla("R09", "Validez", "temperatura_oc", "Temperatura entre 0°C y 45°C", "ALTA",
                 df["temperatura_oc"].between(0, 45), aplicable=df["temperatura_oc"].notna())

registrar_regla("R10", "Validez", "humedad_relativa", "Humedad relativa entre 0% y 100%", "ALTA",
                 df["humedad_relativa"].between(0, 100), aplicable=df["humedad_relativa"].notna())

registrar_regla("R11", "Validez", "precipitacion_mm", "Precipitación no puede ser negativa", "MEDIA",
                 df["precipitacion_mm"] >= 0, aplicable=df["precipitacion_mm"].notna())

registrar_regla("R12", "Validez", "grados_dia_oc", "Grados-día del cultivo no puede ser negativo", "MEDIA",
                 df["grados_dia_oc"] >= 0, aplicable=df["grados_dia_oc"].notna())

print("✓ Reglas R08–R12 ejecutadas.")

## 9. Consistencia con la trazabilidad de la Semana 6
A diferencia del caso guía (que anula el valor cuando el flag es verdadero), en este proyecto decidimos **no borrar valores atípicos** — solo marcarlos, porque en el análisis fitosanitario un valor extremo puede ser la señal real que buscamos, no ruido. Por eso la regla de consistencia aquí es distinta: verificamos que **la bandera coincida exactamente con la condición de rango**, no que el valor haya sido anulado.

In [ ]:
registrar_regla("R13", "Consistencia", "temperatura_oc + flag_temperatura_imposible",
    "flag_temperatura_imposible debe ser True solo cuando la temperatura está fuera de [0,45]", "CRÍTICA",
    df["flag_temperatura_imposible"] == ~df["temperatura_oc"].between(0, 45))

registrar_regla("R14", "Consistencia", "humedad_relativa + flag_humedad_imposible",
    "flag_humedad_imposible debe ser True solo cuando la humedad está fuera de [0,100]", "CRÍTICA",
    df["flag_humedad_imposible"] == ~df["humedad_relativa"].between(0, 100))

print("✓ Reglas R13–R14 ejecutadas.")

## 10. Avisos de revisión
Un atípico estadístico no es automáticamente un error: se deja como aviso para revisión de contexto.

In [ ]:
q1, q3 = df["temperatura_oc"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["flag_temperatura_atipica"] = ~df["temperatura_oc"].between(limite_inf, limite_sup)

AVISOS = {
    "A01": {"dimension": "Revisión", "campo": "temperatura_oc",
             "descripcion": "Posible atípico de temperatura (regla IQR); contrastar con evidencia de campo",
             "severidad": "AVISO", "flag": "flag_temperatura_atipica"},
}

resumen_reglas_df = pd.DataFrame(resumen_reglas)
display(resumen_reglas_df[["regla","dimension","campo","severidad","aplicables","incumplen","cumplimiento_pct","estado_regla"]])

## 11. Clasificación del estado de calidad por registro

In [ ]:
columnas_criticas_altas = [m["col_error"] for m in metadatos_reglas.values() if m["severidad"] in {"CRÍTICA","ALTA"}]
columnas_medias = [m["col_error"] for m in metadatos_reglas.values() if m["severidad"] == "MEDIA"]
columnas_avisos = [a["flag"] for a in AVISOS.values()]

tiene_falla_mayor = df[columnas_criticas_altas].any(axis=1)
tiene_observacion = df[columnas_medias].any(axis=1) | df[columnas_avisos].any(axis=1)

df["estado_calidad"] = np.select([tiene_falla_mayor, tiene_observacion], ["NO CONFORME","OBSERVADO"], default="ACEPTADO")

todas_columnas_error = [m["col_error"] for m in metadatos_reglas.values()]
df["n_incumplimientos"] = df[todas_columnas_error].sum(axis=1)
df["n_avisos"] = df[columnas_avisos].sum(axis=1)

display(df["estado_calidad"].value_counts(dropna=False).to_frame("registros"))

## 12. Tabla larga de incidencias

In [ ]:
def valor_observado(fila, campo):
    if " + " in campo:
        return ""
    return fila.get(campo, "")

incidencias = []
for idx, fila in df.iterrows():
    for rule_id, meta in metadatos_reglas.items():
        if bool(fila[meta["col_error"]]):
            incidencias.append({
                "fila": idx, "fecha": fila.get("fecha", pd.NA), "regla": rule_id,
                "dimension": meta["dimension"], "campo": meta["campo"], "severidad": meta["severidad"],
                "descripcion": meta["descripcion"], "valor_observado": valor_observado(fila, meta["campo"]),
                "estado_calidad": fila["estado_calidad"],
            })
    for aviso_id, aviso in AVISOS.items():
        if bool(fila.get(aviso["flag"], False)):
            incidencias.append({
                "fila": idx, "fecha": fila.get("fecha", pd.NA), "regla": aviso_id,
                "dimension": aviso["dimension"], "campo": aviso["campo"], "severidad": aviso["severidad"],
                "descripcion": aviso["descripcion"], "valor_observado": fila.get(aviso["campo"], ""),
                "estado_calidad": fila["estado_calidad"],
            })

incidencias_df = pd.DataFrame(incidencias)
print("Total incidencias:", len(incidencias_df))
display(incidencias_df.head(10))

## 13. Métricas globales por dimensión

In [ ]:
campos_requeridos = ["fecha", "temperatura_oc", "humedad_relativa"]
completitud_pct = df[campos_requeridos].notna().sum().sum() / (len(df) * len(campos_requeridos)) * 100

fechas_presentes = df["fecha"].dropna()
unicidad_pct = fechas_presentes.nunique() / len(fechas_presentes) * 100 if len(fechas_presentes) else np.nan

def metrica_dimension(nombre):
    sub = resumen_reglas_df[resumen_reglas_df["dimension"] == nombre]
    total_aplica = sub["aplicables"].sum()
    total_cumple = sub["cumplen"].sum()
    return total_cumple / total_aplica * 100 if total_aplica else np.nan

metricas_calidad = pd.DataFrame({
    "metrica": ["Completitud campos obligatorios", "Unicidad de fecha", "Validez", "Consistencia"],
    "valor_pct": [completitud_pct, unicidad_pct, metrica_dimension("Validez"), metrica_dimension("Consistencia")],
})
display(metricas_calidad)

## 14. Plan de acción

In [ ]:
conteo_severidad = incidencias_df["severidad"].value_counts() if not incidencias_df.empty else pd.Series(dtype="int64")

plan_accion = pd.DataFrame([
    {"severidad": "CRÍTICA", "incidencias": int(conteo_severidad.get("CRÍTICA", 0)),
     "accion": "Contrastar con la exportación original de la estación. Corregir si hay evidencia; si no, excluir del análisis y documentar.",
     "prioridad": "Inmediata"},
    {"severidad": "ALTA", "incidencias": int(conteo_severidad.get("ALTA", 0)),
     "accion": "Corregir prioritariamente o limitar el uso del registro hasta resolver.",
     "prioridad": "Alta"},
    {"severidad": "MEDIA", "incidencias": int(conteo_severidad.get("MEDIA", 0)),
     "accion": "Revisar, documentar y decidir si el análisis puede continuar con la limitación explícita.",
     "prioridad": "Programada"},
    {"severidad": "AVISO", "incidencias": int(conteo_severidad.get("AVISO", 0)),
     "accion": "Registrar como contexto; no requiere acción inmediata salvo confirmación de campo.",
     "prioridad": "Informativa"},
])
display(plan_accion)

## 15. Exportar evidencia reproducible

In [ ]:
ARCHIVO_VALIDADO = "clima_validado_semana7.csv"
ARCHIVO_INCIDENCIAS = "incidencias_validacion_clima.csv"
ARCHIVO_REPORTE = "reporte_validacion_clima.xlsx"

df.to_csv(ARCHIVO_VALIDADO, index=False, encoding="utf-8-sig")
incidencias_df.to_csv(ARCHIVO_INCIDENCIAS, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ARCHIVO_REPORTE, engine="openpyxl") as writer:
    resumen_reglas_df.to_excel(writer, sheet_name="Resumen_Reglas", index=False)
    metricas_calidad.to_excel(writer, sheet_name="Metricas_Calidad", index=False)
    incidencias_df.to_excel(writer, sheet_name="Incidencias", index=False)
    df.to_excel(writer, sheet_name="Dataset_Validado", index=False)
    plan_accion.to_excel(writer, sheet_name="Plan_Accion", index=False)

print("Archivos generados:")
print(" -", ARCHIVO_VALIDADO)
print(" -", ARCHIVO_INCIDENCIAS)
print(" -", ARCHIVO_REPORTE)

## 16. Decisión final del caso

In [ ]:
fallas_criticas = resumen_reglas_df[(resumen_reglas_df["severidad"]=="CRÍTICA") & (resumen_reglas_df["estado_regla"]=="NO CUMPLE")]
fallas_altas = resumen_reglas_df[(resumen_reglas_df["severidad"]=="ALTA") & (resumen_reglas_df["estado_regla"]=="NO CUMPLE")]

if len(fallas_criticas) > 0:
    estado_dataset = "REQUIERE CORRECCIÓN ANTES DEL ANÁLISIS"
elif len(fallas_altas) > 0:
    estado_dataset = "APTO SOLO CON CORRECCIONES PRIORITARIAS"
elif (df["estado_calidad"] == "OBSERVADO").any():
    estado_dataset = "APTO CON OBSERVACIONES DOCUMENTADAS"
else:
    estado_dataset = "APTO SEGÚN LAS REGLAS INTERNAS DEFINIDAS"

print("ESTADO DEL DATASET DE CLIMA:")
print(estado_dataset)